# Regression Metrics: A Comprehensive Guide

Evaluating a regression model is just as important as training it. If you don't use the correct metric, you can easily trick yourself into believing a model is performing flawlessly when it is failing catastrophically on critical edge cases.

Here is the complete, comprehensive breakdown of the five primary regression metrics, structured with formulas, structural examples, and their trade-offs.

## 1. MAE (Mean Absolute Error)

### The Theory
MAE measures the average magnitude of errors in a set of predictions, without considering their direction. It is the average horizontal or vertical distance between each actual data point and the regression line.

### The Formula
$$\text{MAE} = \frac{1}{n} \sum_{i=1}^n |y_i - \hat{y}_i|$$
Where $y_i$ is the actual value, $\hat{y}_i$ is the predicted value, and $n$ is the total number of samples.

### Concrete Example
Imagine predicting house prices (in Lakhs):

Actual: `[20, 30, 50]` | Predicted: `[18, 35, 50]`

Errors: `[20-18, 30-35, 50-50]` $\rightarrow$ `[2, -5, 0]`

Absolute Errors: `[2, 5, 0]`

$\text{MAE} = \frac{2 + 5 + 0}{3} = 2.33\text{ Lakhs}$

### Advantages
*   **Intuitive & Unit-Consistent:** The error value is in the exact same units as the target variable (e.g., Lakhs, LPA, CGPA).
*   **Robust to Outliers:** Because it uses absolute differences rather than squares, it treats all errors linearly. A massive outlier won't disproportionately distort the score.

### Disadvantages
*   **Mathematical Discontinuity:** The absolute value function creates a sharp "V" shape at zero. This means it is not differentiable at $e=0$, making it harder to use directly as a loss function in gradient-based optimization without approximations.

## 2. MSE (Mean Squared Error)

### The Theory
MSE measures the average of the squares of the errors. By squaring the error before averaging, it alters the scale and changes how the metric responds to mistakes.

### The Formula
$$\text{MSE} = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2$$

### Concrete Example
Using the same house prices:

Errors: `[2, -5, 0]`

Squared Errors: `[4, 25, 0]`

$\text{MSE} = \frac{4 + 25 + 0}{3} = 9.66\text{ (Lakhs)}^2$

### Advantages
*   **Differentiable:** The squared function forms a smooth parabola, creating an ideal surface for optimization algorithms like Gradient Descent.
*   **Severe Outlier Penalty:** It highlights large errors. If your model misses a prediction by 10 units, the penalty is 100. If it misses by 20, the penalty jumps to 400.

### Disadvantages
*   **Unit Distortion:** The final unit is squared (e.g., $\text{Lakhs}^2$), making it impossible to interpret directly alongside the raw data.
*   **Outlier Vulnerability:** A single extreme outlier can completely ruin the MSE score, even if the model performs perfectly on the other 99% of the dataset.

## 3. RMSE (Root Mean Squared Error)

### The Theory
RMSE builds directly on top of MSE by taking the square root of the final average squared error. This brings the metric's scale back to the original units of the target variable while preserving the outlier penalty properties of MSE.

### The Formula
$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2}$$

### Concrete Example
$\text{MSE} = 9.66$

$\text{RMSE} = \sqrt{9.66} = 3.11\text{ Lakhs}$

### Advantages
*   **Unit-Consistent:** Combines the unit interpretability of MAE with the mathematical benefits of MSE.
*   **Maintains Outlier Awareness:** It still penalizes larger errors more heavily than smaller errors, making it a reliable choice when large mistakes are highly undesirable.

### Disadvantages
*   **Sensitive to Scale:** Like MSE, it is heavily influenced by outliers, which can make a model look poor overall due to a few exceptional data points.

## 4. R² Score (Coefficient of Determination)

### The Theory
Unlike MAE, MSE, and RMSE, which tell you the absolute distance or scale of your mistakes, the $R^2$ score measures relative performance. It determines how much better your model is compared to a baseline model that simply predicts the mean (average) of the target variable every single time.

### The Formula
$$R^2 = 1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}}$$

$\text{Where } SS_{\text{res}} = \sum_{i=1}^n (y_i - \hat{y}_i)^2 \quad \text{(Residual Sum of Squares - Your Model's Mistakes)}$

$SS_{\text{tot}} = \sum_{i=1}^n (y_i - \bar{y})^2 \quad \text{(Total Sum of Squares - Baseline Mean Model's Mistakes)}$

$R^2 = 1$: Perfect model. Your residuals are zero.

$R^2 = 0$: Your model is performing exactly as well as a dumb baseline model predicting the mean.

$R^2 < 0$: Your model is performing worse than simply predicting the average value for every row.

### Advantages
*   **Scale-Independent:** It outputs a relative score (typically between 0 and 1), allowing you to compare models trained on completely different datasets (e.g., comparing a model predicting CGPA to one predicting LPA).

### Disadvantages
*   **The False Progression Trap:** $R^2$ will never decrease when you add new features to your dataset, even if those features are complete garbage (like adding a student's favorite color to predict their placement LPA). The model can use random noise in the new feature to slightly improve its training fit, artificially inflating the score.

## 5. Adjusted R² Score

### The Theory
Adjusted $R^2$ is the direct fix for the false progression trap of standard $R^2$. It penalizes the score based on the number of features added to the model. If a newly added feature does not improve the model's predictive power by a significant margin, the Adjusted $R^2$ score drops.

### The Formula
$$\text{Adjusted } R^2 = 1 - \left[ \frac{(1 - R^2)(n - 1)}{n - p - 1} \right]$$
Where $n$ is the number of data points (samples) and $p$ is the number of independent features (predictors).

### Advantages
*   **True Feature Evaluation:** It accurately indicates whether adding a new feature actually adds predictive value or just introduces noise.
*   **Prevents Overfitting:** Helps guard against adding too many dimensions, protecting models from the Curse of Dimensionality.

### Disadvantages
*   **Can turn Negative:** If the model includes many useless predictors, the score can drop significantly below zero, which can be unintuitive to interpret initially.

# Let's see those in code

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# creating a dummy data
np.random.seed(42)
n_samples = 1000

# Feature 1: The Standard Indicator
cgpa = np.random.normal(7.0, 1.5, n_samples)

# Feature 2: The Garbage Feature (Zero correlation)
shoe_size = np.random.randint(5, 13, n_samples)

# Feature 3: The Secret Weapon (High correlation)
coding_score = np.random.normal(75, 10, n_samples)

# Target Variable: LPA depends on CGPA and Coding Score, NOT Shoe Size
noise = np.random.normal(0, 1.5, n_samples)
lpa = (cgpa * 1.5) + (coding_score * 0.1) - 4 + noise

df = pd.DataFrame({
    'CGPA': cgpa,
    'Shoe_Size': shoe_size,
    'Coding_Score': coding_score,
    'LPA': lpa
})

In [4]:
# Helper function to quickly train and return all metrics
def train_and_evaluate(feature_list, target='LPA'):
  X=df[feature_list]
  y=df[target]
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

  # Train the model
  model = LinearRegression()
  model.fit(X_train,y_train)
  y_pred = model.predict(X_test)

  # calculate the base metrics
  mae = mean_absolute_error(y_test, y_pred)
  mse = mean_squared_error(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)

  # adjusted R2
  n = len(X_test)      # no. of test samples
  p = X_test.shape[1]  # no. of features used
  adj_r2 = 1 - ((1-r2) * (n-1) / (n-p-1))

  return mae, mse, rmse, r2, adj_r2

In [5]:
# test1: the base model CGPA only
print("--- TEST 1: BASE MODEL (CGPA Only) ---")
mae_1, mse_1, rmse_1, r2_1, adj_r2_1 = train_and_evaluate(['CGPA'])

print(f"MAE:          {mae_1:.4f} Lakhs")
print(f"MSE:          {mse_1:.4f} (Lakhs squared)")
print(f"RMSE:         {rmse_1:.4f} Lakhs")
print(f"R2 Score:     {r2_1:.6f}")
print(f"Adjusted R2:  {adj_r2_1:.6f}")

--- TEST 1: BASE MODEL (CGPA Only) ---
MAE:          1.3385 Lakhs
MSE:          2.9021 (Lakhs squared)
RMSE:         1.7036 Lakhs
R2 Score:     0.594355
Adjusted R2:  0.592306


In [6]:
# test2: adding a garbage feature
print("\n--- TEST 2: ADDING 'Shoe_Size' (The Trap) ---")
_, _, _, r2_2, adj_r2_2 = train_and_evaluate(['CGPA', 'Shoe_Size'])

print(f"Old R2:       {r2_1:.6f}  |  New R2:       {r2_2:.6f}")
print(f"Old Adj R2:   {adj_r2_1:.6f}  |  New Adj R2:   {adj_r2_2:.6f}")

if r2_2 >= r2_1:
    print("⚠️ DANGER: Standard R2 went UP (or stayed flat) even though Shoe Size is useless!")
if adj_r2_2 < adj_r2_1:
    print("🛡️ SAVED: Adjusted R2 went DOWN, correctly penalizing the garbage feature!")


--- TEST 2: ADDING 'Shoe_Size' (The Trap) ---
Old R2:       0.594355  |  New R2:       0.594002
Old Adj R2:   0.592306  |  New Adj R2:   0.589881
🛡️ SAVED: Adjusted R2 went DOWN, correctly penalizing the garbage feature!


In [7]:
# Test3: adding that important feature
print("\n--- TEST 3: ADDING 'Coding_Score' (The Real Upgrade) ---")
# Now we test with all three features
_, _, _, r2_3, adj_r2_3 = train_and_evaluate(['CGPA', 'Shoe_Size', 'Coding_Score'])

print(f"Previous R2:      {r2_2:.6f}  |  New R2:       {r2_3:.6f}")
print(f"Previous Adj R2:  {adj_r2_2:.6f}  |  New Adj R2:   {adj_r2_3:.6f}")

print("✅ SUCCESS: Both R2 and Adjusted R2 skyrocketed because the new feature actually held predictive power!")


--- TEST 3: ADDING 'Coding_Score' (The Real Upgrade) ---
Previous R2:      0.594002  |  New R2:       0.687315
Previous Adj R2:  0.589881  |  New Adj R2:   0.682529
✅ SUCCESS: Both R2 and Adjusted R2 skyrocketed because the new feature actually held predictive power!
